# 03 - Train MLP from Pre-extracted Concerto Features

This notebook trains the translation head on the pre-extracted S3DIS Area 5 features. It is designed to run on a Google Colab T4 instance.

Ensure you have access to the shared Google Drive with the preprocessed data and extracted features.

### 1. Setup & Mount Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/Deep_learning_project'
DRIVE_ROOT = '/content/drive/MyDrive/DL_Project'
DRIVE_FEATURES_DIR = f'{DRIVE_ROOT}/features/s3dis_area5'
LOCAL_FEATURES_ROOT = '/content/local_features'
LOCAL_FEATURES_DIR = f'{LOCAL_FEATURES_ROOT}/s3dis_area5'
BRANCH = 'colab-setups'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_DIR}


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
%cd {REPO_DIR}
!git fetch origin
!git checkout -B {BRANCH} origin/{BRANCH}
!git pull origin {BRANCH}
!git branch --show-current
!git log -1 --oneline

/content/Deep_learning_project
Branch 'dev/enc-mlp' set up to track remote branch 'dev/enc-mlp' from 'origin'.
Switched to a new branch 'dev/enc-mlp'
From https://github.com/Gandata/Deep_learning_project
 * branch            dev/enc-mlp -> FETCH_HEAD
Already up to date.
dev/enc-mlp
450c78d (HEAD -> dev/enc-mlp, origin/dev/enc-mlp) undo changes


In [15]:
%cd /content/Deep_learning_project
!git restore configs/train_mlp_s3dis.yaml notebooks/pyproject.toml
!git fetch origin
!git checkout -B colab-setups origin/colab-setups
!git pull --no-edit origin colab-setups
!git branch --show-current
!git log -1 --oneline


/content/Deep_learning_project
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.11 KiB | 568.00 KiB/s, done.
From https://github.com/Gandata/Deep_learning_project
   04cdbfb..4b92e3a  colab-setups -> origin/colab-setups
Branch 'colab-setups' set up to track remote branch 'colab-setups' from 'origin'.
Switched to a new branch 'colab-setups'
From https://github.com/Gandata/Deep_learning_project
 * branch            colab-setups -> FETCH_HEAD
Already up to date.
colab-setups
4b92e3a (HEAD -> colab-setups, origin/colab-setups) fix(colab-setups): train from local copied features and remove pre-inspection


In [5]:
!git clone https://github.com/Pointcept/Concerto.git /content/Concerto

Cloning into '/content/Concerto'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 74 (delta 10), reused 23 (delta 9), pack-reused 42 (from 1)
Receiving objects: 100% (74/74), 2.12 MiB | 52.95 MiB/s, done.
Resolving deltas: 100% (14/14), done.


### 2. Symlink Data & Checkpoints
We map Drive data/checkpoints into the repository, but copy training features to local runtime storage so training is not bottlenecked by the Drive mount.

In [6]:
# Symlink Drive data/checkpoints and copy features locally for faster training
!rm -f data checkpoints features pretrained
!ln -sf {DRIVE_ROOT}/data ./data
!ln -sf {DRIVE_ROOT}/checkpoints ./checkpoints
!mkdir -p {LOCAL_FEATURES_ROOT}
!rm -rf {LOCAL_FEATURES_DIR}
!cp -r {DRIVE_FEATURES_DIR} {LOCAL_FEATURES_ROOT}/
!ln -sf {LOCAL_FEATURES_ROOT} ./features
!readlink -f ./features/s3dis_area5
!du -sh {LOCAL_FEATURES_DIR}

/content/local_features/s3dis_area5
31G	/content/local_features/s3dis_area5


In [7]:
%cd notebooks/

/content/Deep_learning_project/notebooks


### 3. Setup Hugging Face Token
Input your Hugging Face token below if `open_clip_torch` needs it to download the models.

In [8]:
# If token is in colab secrets

from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

# Write to .env
with open(".env", "w") as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")

### 4. Validate Inputs and Run Training
This section checks the extracted `.npz` files, fixes the config paths for Colab, and then launches MLP training.

In [9]:
# Forzar a uv a usar/instalar Python 3.10.12 y sincronizar
!uv python install 3.10.12
!uv sync --python 3.10.12

Installed Python 3.10.12 in 2.78s
 + cpython-3.10.12-linux-x86_64-gnu (python3.10)
Using CPython 3.10.12
Creating virtual environment at: .venv
Resolved 147 packages in 6.54s
Prepared 140 packages in 55.75s
Installed 141 packages in 375ms
 + addict==2.4.0
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + anyio==4.13.0
 + attrs==26.1.0
 + blinker==1.9.0
 + brotli==1.2.0
 + camtools==0.1.8
 + ccimport==0.4.4
 + certifi==2026.4.22
 + charset-normalizer==3.4.7
 + click==8.3.3
 + configargparse==1.7.5
 + contourpy==1.3.2
 + cumm-cu120==0.4.11
 + cycler==0.12.1
 + dash==4.1.0
 + dotenv==0.9.9
 + exceptiongroup==1.3.1
 + fast-pytorch-kmeans==0.2.2
 + fastapi==0.136.1
 + fastjsonschema==2.21.2
 + filelock==3.29.0
 + fire==0.7.1
 + flask==3.1.3
 + fonttools==4.62.1
 + fsspec==2026.4.0
 + ftfy==6.3.1
 + gradio==6.14.0
 + gradio-client==2.5.0
 + groovy==0.1.2
 + h11==0.16.0
 + hf-gradio==0.4.1
 + hf-xet==1.4.3
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.13.0
 + idna==3.13
 + imag

In [10]:
!uv add dotenv
!uv add open_clip_torch

Resolved 147 packages in 1ms
Checked 141 packages in 2ms
Resolved 147 packages in 1ms
Checked 141 packages in 1ms


In [25]:
%cd /content/Deep_learning_project
!git restore configs/train_mlp_s3dis.yaml notebooks/pyproject.toml
!git fetch origin
!git checkout -B colab-setups origin/colab-setups
!git pull --no-edit origin colab-setups
!grep -n "epochs:\|batch_size:\|save_every:" /content/Deep_learning_project/configs/train_mlp_s3dis.yaml



/content/Deep_learning_project
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 401 bytes | 401.00 KiB/s, done.
From https://github.com/Gandata/Deep_learning_project
   ebf7225..22ae670  colab-setups -> origin/colab-setups
Branch 'colab-setups' set up to track remote branch 'colab-setups' from 'origin'.
Reset branch 'colab-setups'
Your branch is up to date with 'origin/colab-setups'.
From https://github.com/Gandata/Deep_learning_project
 * branch            colab-setups -> FETCH_HEAD
Already up to date.
40:  epochs: 40
41:  batch_size: 16384
48:  save_every: 1


In [26]:
import yaml
import os

# Define the file path
file_path = '/content/Deep_learning_project/configs/train_mlp_s3dis.yaml'
base_prefix = '/content/Deep_learning_project/'

# 1. Load the existing YAML content
with open(file_path, 'r') as f:
    config = yaml.safe_load(f)

# 2. Update the paths
# Update 'data' section
config['data']['train_features_path'] = os.path.join(base_prefix, config['data']['train_features_path'])
config['data']['label_embeddings_path'] = os.path.join(base_prefix, config['data']['label_embeddings_path'])

# Update 'training' section
config['training']['checkpoint_dir'] = os.path.join(base_prefix, config['training']['checkpoint_dir'])
config['training']['metrics_path'] = os.path.join(base_prefix, config['training']['metrics_path'])

# 3. Save the modified YAML back to the file
with open(file_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"Successfully updated paths in {file_path}")

Successfully updated paths in /content/Deep_learning_project/configs/train_mlp_s3dis.yaml


In [27]:
# Run the real extraction script with chunking to avoid T4 OOM on large rooms
#!PYTHONPATH=/content/Concerto uv run python /content/Deep_learning_project/scripts/extract_features.py --data_dir /content/Deep_learning_project/data/s3dis_processed --out_dir /content/Deep_learning_project/features/s3dis_area5 --feature-dtype float16 --max-points-per-chunk 100000
!PYTHONPATH=/content/Deep_learning_project:/content/Concerto uv run python /content/Deep_learning_project/src/train.py --config /content/Deep_learning_project/configs/train_mlp_s3dis.yaml


Using device: cuda
Train files:   68
Feature dim:   896 (from config)
Model params:  984,576
Epoch 001 | train_loss=0.000068
Epoch 002 | train_loss=0.000030
Epoch 003 | train_loss=0.000020
Epoch 004 | train_loss=0.000017
Epoch 005 | train_loss=0.000015
Epoch 006 | train_loss=0.000014
Epoch 007 | train_loss=0.000012
Epoch 008 | train_loss=0.000012
Epoch 009 | train_loss=0.000010
Epoch 010 | train_loss=0.000010
Epoch 011 | train_loss=0.000010
Epoch 012 | train_loss=0.000009
Epoch 013 | train_loss=0.000009
Epoch 014 | train_loss=0.000008
Epoch 015 | train_loss=0.000008
Epoch 016 | train_loss=0.000007
Epoch 017 | train_loss=0.000007
Epoch 018 | train_loss=0.000007
Epoch 019 | train_loss=0.000007
Epoch 020 | train_loss=0.000007
Epoch 021 | train_loss=0.000006
Epoch 022 | train_loss=0.000006
Epoch 023 | train_loss=0.000006
Epoch 024 | train_loss=0.000006
Epoch 025 | train_loss=0.000006
Epoch 026 | train_loss=0.000006
Epoch 027 | train_loss=0.000005
Epoch 028 | train_loss=0.000005
Epoch 029 |

auto: notebook execution

# Other tests

In [13]:
#!PYTHONPATH=/content/Concerto uv run python /content/Concerto/demo/0_pca.py

In [14]:
#!PYTHONPATH=/content/Concerto uv run python -c "import sys, torch; sys.path.insert(0, '/content/Concerto'); import concerto; import spconv.pytorch as spconv; print('python ok'); print(torch.__version__, torch.version.cuda); print(spconv.__file__)"